### Decorators for benchmarking

Inside decorators, `perf_counter()` is the canonical tool.

In [1]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.6f} seconds")
        return result
    return wrapper

### Decorator for aggregation of **recursive functions**

In [ ]:
def timer(func):
    depth = 0  # shared state for this function only

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal depth
        is_top_level = depth == 0
        depth += 1

        if is_top_level:
            start = time.perf_counter()

        result = func(*args, **kwargs)  # direct call to original fib

        depth -= 1

        if is_top_level:
            end = time.perf_counter()
            print(f"{func.__name__} took {end - start:.6f} seconds")

        return result
    return wrapper

In [ ]:
@timer
def fibonacci(num):
    if num <= 0:
        return 0
    elif num == 1:
        return 1
    # recursive fib call now refers to decorated function (wrapper), since we replaced name at decoration time
    return fibonacci(num - 1) + fibonacci(num - 2)  # here recursion becomes decorated

fibonacci(5)

fibonacci took 0.000008 seconds


5

### What actually happens when decorating a recursive function

#### 1. Syntactic sugar

```py
@timer
def fibonacci(n):
    ...
```
it **immediately rewrites** it to:
```py
fibonacci = timer(fibonacci)
```
This happens **once**, at import time, before any calls.

So after this line runs `fibonacci → wrapper` and the original function is now stored inside the wrapper as `func`.

So:
- `func` inside the wrapper refers to original `fibonacci`
- The global name `fibonacci` refers to `wrapper`

This is the key to everything.

**Important**:
`timer` receives the original function object — the real `fibonacci` — **before any decoration** happens.

---

#### 2. Inside original `fibonacci(5)` after decoration
Original function contains:
```py
return fibonacci(n-1) + fibonacci(n-2)
```
Inside the original function, `fibonacci` refers to the **decorated** function (`wrapper`), not the original one.

Because the name `fibonacci` was `replaced` at decoration time.

So inside the original function:
```shell
fibonacci(4) → wrapper(4)
fibonacci(3) → wrapper(3)
...
```
This is the moment where the recursion **switches** from original → decorated.

---

#### 3. This is the core mechanic

```shell
wrapper(5)
    → original fibonacci(5)
        → wrapper(4)
            → original fibonacci(4)
                → wrapper(3)
                    → original fibonacci(3)
                        ...
```
The decorator is applied **once**, but the `wrapper` is invoked **many times** because the recursive function calls itself by name:
- Decoration happens once.
- Wrapper executes many times.
- Original function executes many times.

---

#### 4. Shared `depth` variable in a decorator

Inside timer, we define:
```py
depth = 0
```
This variable lives in the **closure of the decorator**.

Meaning:
- It is **not global**
- It is **not inside the wrapper**
- It is **not recreated on each call**
- It is **shared by all recursive calls of this one decorated function**

Each decorated function gets its own private `depth`.

If you decorate 10 functions with `@timer`, each one gets its own independent `depth`.

---

#### 5. Why does the depth trick work?
Because:
- Every wrapper call increments `depth`
- Every wrapper exit decrements `depth`
- Only the outermost call sees `depth == 0`
- Therefore only the outermost call prints timing

This is a classic technique for:
- recursion‑aware decorators
- logging
- profiling
- memoization
- tracing

---

#### 6. The lifecycle of `depth`

Let’s walk through a call:

**First call**: `fibonacci(5)`
- `depth = 0`
- `wrapper` runs
- `is_top_level = True`
- `depth` becomes 1
- `wrapper` calls the **original** fibonacci: `func(*args, **kwargs)`

**Inside original fibonacci**:

It calls:
```py
fibonacci(4)
```
But remember: `fibonacci = wrapper`

So this is a **decorated call**, not a direct call.

**Second call**: `fibonacci(4)`
- `depth = 1`
- `wrapper` runs
- `is_top_level = False`
- `depth` becomes 2
- `wrapper` calls original fibonacci

And so on…

When recursion unwinds:
- Each `wrapper` call finishes
- `depth -= 1`
- Eventually `depth` returns to 0
- Only then do we print the timing

This is why we get **one aggregated measurement**.

---

#### 7. Final Mental Model
```py
@timer
def fibonacci(n):
    ...
```

1. `timer` receives original `fibonacci`
2. `timer` creates `wrapper` + `depth` counter
3. `fibonacci = wrapper`

Calling `fibonacci(5)`:

```

wrapper(5)
    depth = 1
    original fibonacci(5)
        fibonacci(4) → wrapper(4)
            depth = 2
            original fibonacci(4)
                fibonacci(3) → wrapper(3)
                    depth = 3
                    ...
    depth returns to 0
    wrapper prints timing
```